<a id="fuzzy-set"></a>

$$
\text{模糊综合评价 (Fuzzy Comprehensive Evaluation): 用模糊数学处理 "好/较好/一般/差" 这类边界不清楚的评价} \\ \quad \\
\text{1. 经典集合 (Crisp Set): 元素要么属于 A, 要么不属于, 用特征函数 (Characteristic Function) 表示} \\
\chi_A(x) = \begin{cases} 1, & x \in A \\ 0, & x \notin A \end{cases} \\
\text{问题: "身高 180 以上算高", 那 179.9 就 "不高" 了? 现实中的概念往往是渐变的} \\ \quad \\
\text{2. 模糊集 (Fuzzy Set): 把特征函数的取值从 \{0, 1\} 推广到区间 [0, 1]} \\
\mu_A(x) \in [0, 1] \text{ 叫隶属度 (Membership Degree), 表示 x 属于 A 的程度} \\ \quad \\
\text{3. 模糊集的运算: 要求在 } \mu \in \{0, 1\} \text{ 时退化成经典集合的运算} \\
\text{经典集合中: } x \in A \cup B \iff \chi_A = 1 \text{ 或 } \chi_B = 1 \iff \max(\chi_A, \chi_B) = 1 \\
\text{同理 } x \in A \cap B \iff \min(\chi_A, \chi_B) = 1 \text{, } x \in A^c \iff 1 - \chi_A = 1 \\
\text{推广到模糊集:} \\
\mu_{A \cup B}(x) = \max(\mu_A, \mu_B) = \mu_A \vee \mu_B, \quad
\mu_{A \cap B}(x) = \min(\mu_A, \mu_B) = \mu_A \wedge \mu_B, \quad
\mu_{A^c}(x) = 1 - \mu_A(x) \\ \quad \\
\text{性质: 德摩根律仍然成立 } 1 - \max(a, b) = \min(1 - a, 1 - b) \\
\text{但排中律不成立: } \mu_A \wedge \mu_{A^c} = \min(\mu, 1 - \mu) \text{ 可以不为 0 (例如 } \mu = 0.5 \text{ 时为 0.5, 既 "高" 又 "不高")}
$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(linewidth=np.inf)
np.set_printoptions(suppress=True)

def plot_nicely():
    for ax in plt.gcf().get_axes():
        ax.grid(True, which="both", alpha=0.3)
        if ax.get_legend_handles_labels()[0]:
            ax.legend(fontsize='small')
    plt.tight_layout()

rng = np.random.default_rng(0)
a, b = rng.random(10000), rng.random(10000)
print(f'德摩根律 1 - max(a, b) == min(1 - a, 1 - b) ? {np.allclose(1 - np.maximum(a, b), np.minimum(1 - a, 1 - b))}')
print(f'德摩根律 1 - min(a, b) == max(1 - a, 1 - b) ? {np.allclose(1 - np.minimum(a, b), np.maximum(1 - a, 1 - b))}')

# 退化成经典集合: 0/1 时 max = 或, min = 且
for x in [0, 1]:
    for y in [0, 1]:
        print(f'chi_A = {x}, chi_B = {y}: max = {max(x, y)} (or = {int(x or y)}), min = {min(x, y)} (and = {int(x and y)})')

mu = np.array([0.0, 0.3, 0.5, 0.8, 1.0])
print(f'mu = {mu}, mu ∧ (1 - mu) = {np.minimum(mu, 1 - mu)} (经典集合中恒为 0)')

<a id="membership-function"></a>

$$
\text{隶属函数 (Membership Function): 给每个 x 一个隶属度} \\ \quad \\
\text{1. 三角形 (Triangular): 在 b 处隶属度为 1, 在 a 和 c 处降为 0, 中间线性变化} \\
\text{过点 (a, 0) 和 (b, 1) 的直线: } \mu = \frac{x - a}{b - a} \text{; 过点 (b, 1) 和 (c, 0) 的直线: } \mu = \frac{c - x}{c - b} \\
\mu(x) = \max\left(\min\left(\frac{x - a}{b - a}, \frac{c - x}{c - b}\right), 0\right) \\ \quad \\
\text{2. 梯形 (Trapezoidal): 在 [b, c] 上隶属度都为 1} \\
\mu(x) = \max\left(\min\left(\frac{x - a}{b - a}, 1, \frac{d - x}{d - c}\right), 0\right) \\ \quad \\
\text{3. 高斯型 (Gaussian): } \mu(x) = e^{-\frac{(x - c)^2}{2\sigma^2}} \text{ (光滑, 但永远不为 0)} \\ \quad \\
\text{4. 偏小型 / 偏大型: 梯形的一侧延伸到无穷, 例如 "冷" 只有右侧下降, "热" 只有左侧上升} \\ \quad \\
\text{例: 温度的三个模糊集 "冷 / 舒适 / 热", 同一个温度可以同时属于两个集合 (例如 18 度: 有点冷, 也有点舒适)}
$$

In [ ]:
def triangular(x, a, b, c):
    return np.maximum(np.minimum((x - a) / (b - a), (c - x) / (c - b)), 0)

def trapezoidal(x, a, b, c, d):
    left = (x - a) / (b - a) if b > a else np.ones_like(x)
    right = (d - x) / (d - c) if d > c else np.ones_like(x)
    return np.clip(np.minimum(np.minimum(left, 1), right), 0, 1)

def gaussian_mf(x, c, sigma):
    return np.exp(-(x - c)**2 / (2 * sigma**2))

T = np.linspace(0, 40, 401)
cold = trapezoidal(T, -np.inf, -np.inf, 12, 20)       # 偏小型: 12 度以下完全是冷
comfortable = triangular(T, 15, 22, 29)
hot = trapezoidal(T, 24, 32, np.inf, np.inf)           # 偏大型

# 独立验证: 三角形/梯形就是分段线性插值
print(f'三角形 == np.interp ? {np.allclose(comfortable, np.interp(T, [15, 22, 29], [0, 1, 0]))}')
print(f'偏小型梯形 == np.interp ? {np.allclose(cold, np.interp(T, [12, 20], [1, 0]))}')
print(f'偏大型梯形 == np.interp ? {np.allclose(hot, np.interp(T, [24, 32], [0, 1]))}')
for t in [10, 18, 22, 27, 35]:
    i = np.argmin(np.abs(T - t))
    print(f'{t} 度: 冷 = {cold[i]:.3f}, 舒适 = {comfortable[i]:.3f}, 热 = {hot[i]:.3f}')

plt.figure(figsize=(4, 3))
plt.plot(T, cold, label='cold (trapezoid, left)')
plt.plot(T, comfortable, label='comfortable (triangle)')
plt.plot(T, hot, label='hot (trapezoid, right)')
plt.plot(T, gaussian_mf(T, 22, 4), '--', label='comfortable (Gaussian)')
plt.xlabel('temperature')
plt.ylabel('membership')
plot_nicely()
plt.show()

<a id="fuzzy-evaluation"></a>

$$
\text{一级模糊综合评价} \\ \quad \\
\text{1. 因素集 (Factor Set) } U = \{u_1, \cdots, u_m\} \text{: 评价时考虑的 m 个指标} \\
\text{2. 评语集 (Comment Set) } V = \{v_1, \cdots, v_k\} \text{: 例如 \{优, 良, 中, 差\}} \\
\text{3. 单因素评价矩阵 (Fuzzy Relation Matrix) R: 第 i 行是只看因素 } u_i \text{ 时, 对象属于每个评语的隶属度} \\
R = \begin{bmatrix} r_{11} & \cdots & r_{1k} \\ \vdots & & \vdots \\ r_{m1} & \cdots & r_{mk} \end{bmatrix} \quad (m \times k) \\
\text{来源: 问卷 (100 个人中 40 人评 "优", 则 } r = 0.4 \text{), 或者用隶属函数把指标值转换成隶属度} \\
\text{4. 权重向量 } A = (a_1, \cdots, a_m), \sum a_i = 1 \text{ (来自 AHP, 熵权法等)} \\
\text{5. 合成: } B = A \circ R = (b_1, \cdots, b_k) \text{, 再按最大隶属度原则 (Maximum Membership Principle) 选 } \arg\max_j b_j \\ \quad \\
\text{合成算子 (Composition Operator) } \circ \text{ 的几种选择:} \\
M(\wedge, \vee): b_j = \max_i \min(a_i, r_{ij}) \quad (\text{主因素决定型, 只有最突出的因素起作用, 信息丢失多}) \\
M(\cdot, \vee): b_j = \max_i (a_i \cdot r_{ij}) \quad (\text{主因素突出型}) \\
M(\wedge, \oplus): b_j = \min\left(1, \sum_i \min(a_i, r_{ij})\right) \\
M(\cdot, +): b_j = \sum_i a_i r_{ij} \quad (\text{加权平均型, 就是矩阵乘法 } B = AR \text{, 所有因素都起作用, 最常用}) \\ \quad \\
\text{加权平均型的性质: 若 R 的每一行和为 1, 且 } \sum a_i = 1 \text{, 则 B 的各分量和也为 1} \\
\sum_j b_j = \sum_j \sum_i a_i r_{ij} = \sum_i a_i \underbrace{\sum_j r_{ij}}_{= 1} = \sum_i a_i = 1 \\ \quad \\
\text{如果 B 的最大值不突出, 也可以给每个评语打分 } s = (95, 85, 75, 60) \text{, 算加权平均分 } \frac{\sum_j b_j s_j}{\sum_j b_j}
$$

数学基础: [矩阵乘法](../math_basics/linear_algebra.ipynb#matrix-multiplication) · 权重来源: [层次分析法 AHP](analytic_hierarchy_process.ipynb) · [熵权法](entropy_weight_method.ipynb)

In [ ]:
def compose(A, R, operator):
    A = np.asarray(A, dtype=float)
    R = np.asarray(R, dtype=float)
    m, k = R.shape
    B = np.zeros(k)
    for j in range(k):
        if operator == 'min-max':
            B[j] = max(min(A[i], R[i, j]) for i in range(m))
        elif operator == 'prod-max':
            B[j] = max(A[i] * R[i, j] for i in range(m))
        elif operator == 'min-sum':
            B[j] = min(1.0, sum(min(A[i], R[i, j]) for i in range(m)))
        elif operator == 'prod-sum':
            B[j] = sum(A[i] * R[i, j] for i in range(m))
    return B

# 例子: 评价一门课, 因素 = {教学内容, 教学方法, 教学态度, 作业反馈}, 评语 = {优, 良, 中, 差}
# 100 名学生的问卷, 每一行是该因素下选择各评语的比例
comments = ['优', '良', '中', '差']
R1 = np.array([[0.40, 0.35, 0.20, 0.05],
               [0.20, 0.45, 0.25, 0.10],
               [0.55, 0.30, 0.10, 0.05],
               [0.10, 0.30, 0.40, 0.20]])
A1 = np.array([0.35, 0.30, 0.15, 0.20])

for op in ['min-max', 'prod-max', 'min-sum', 'prod-sum']:
    B = compose(A1, R1, op)
    print(f'{op:<9}: B = {np.round(B, 4)}, 最大隶属度 -> {comments[np.argmax(B)]}')

# 独立验证 (向量化写法)
print(f'prod-sum == A @ R ? {np.allclose(compose(A1, R1, "prod-sum"), A1 @ R1)}')
print(f'min-max == max(min(A[:, None], R)) ? {np.allclose(compose(A1, R1, "min-max"), np.max(np.minimum(A1[:, None], R1), axis=0))}')
print(f'prod-sum 的 B 各分量和 = {np.sum(A1 @ R1):.4f}')

scores = np.array([95, 85, 75, 60])
B1 = A1 @ R1
print(f'加权平均分 = {B1 @ scores / B1.sum():.2f}')

# M(∧, ∨) 丢失信息: "作业反馈" 变得很差, 但它的隶属度被权重 0.2 截断 (min(0.2, 0.7) = 0.2), 结果完全不变
R1_bad = R1.copy()
R1_bad[3] = [0.0, 0.1, 0.2, 0.7]
print(f'作业反馈变差后 min-max: {np.round(compose(A1, R1_bad, "min-max"), 4)} (和原来一样), '
      f'prod-sum: {np.round(A1 @ R1_bad, 4)} ("差" 从 0.095 升到 {(A1 @ R1_bad)[3]:.3f})')

<a id="multi-level-fuzzy"></a>

$$
\text{多级模糊综合评价 (Multi-Level): 因素太多时, 先分组评价, 再把各组的结果作为上一级的单因素评价} \\ \quad \\
\text{把因素集分成 s 组: } U = U_1 \cup U_2 \cup \cdots \cup U_s \\
\text{第一级: 每组单独评价 } B_t = A_t \circ R_t \quad (t = 1, \cdots, s) \\
\text{第二级: 把 } B_1, \cdots, B_s \text{ 叠成新的评价矩阵, 组之间的权重为 A} \\
R = \begin{bmatrix} B_1 \\ B_2 \\ \vdots \\ B_s \end{bmatrix}, \quad B = A \circ R \\ \quad \\
\text{原因: 如果一次放入很多因素, 每个权重 } a_i \text{ 都会很小, 在 } M(\wedge, \vee) \text{ 下 } \min(a_i, r_{ij}) \text{ 几乎都等于 } a_i \text{, 评价失去区分度} \\
\text{(用加权平均算子时, 两级评价等价于把组权重和组内权重相乘后的一级评价: } B = \sum_t A_t^{(2)} A_t R_t \text{)}
$$

In [ ]:
# 评价一个教师: 3 组因素
# 组 1 教学 (内容, 方法), 组 2 科研 (论文, 项目, 获奖), 组 3 服务 (指导学生, 公共服务)
R_teach = np.array([[0.5, 0.3, 0.2, 0.0],
                    [0.3, 0.4, 0.2, 0.1]])
A_teach = np.array([0.6, 0.4])
R_research = np.array([[0.2, 0.3, 0.3, 0.2],
                       [0.1, 0.4, 0.4, 0.1],
                       [0.0, 0.2, 0.5, 0.3]])
A_research = np.array([0.5, 0.3, 0.2])
R_service = np.array([[0.6, 0.3, 0.1, 0.0],
                      [0.3, 0.4, 0.2, 0.1]])
A_service = np.array([0.7, 0.3])
A_top = np.array([0.5, 0.3, 0.2])

level1 = [A_t @ R_t for A_t, R_t in [(A_teach, R_teach), (A_research, R_research), (A_service, R_service)]]
R_top = np.vstack(level1)
B_top = A_top @ R_top
print(f'第一级结果 (每一行是一组的 B_t) = \n{np.round(R_top, 4)}')
print(f'第二级结果 B = {np.round(B_top, 4)}, 评价: {comments[np.argmax(B_top)]}, 加权平均分 = {B_top @ scores / B_top.sum():.2f}')

# 验证: 加权平均算子下, 两级 == 把权重相乘后的一级评价
A_flat = np.concatenate([A_top[0] * A_teach, A_top[1] * A_research, A_top[2] * A_service])
R_flat = np.vstack([R_teach, R_research, R_service])
print(f'两级 == 一级 (权重相乘) ? {np.allclose(B_top, A_flat @ R_flat)}, 一级权重和 = {A_flat.sum():.4f}')

plt.figure(figsize=(4, 3))
x_pos = np.arange(4)
for t, (name, Bt) in enumerate(zip(['teaching', 'research', 'service'], level1)):
    plt.bar(x_pos + (t - 1.5) * 0.2, Bt, width=0.2, label=name)
plt.bar(x_pos + 1.5 * 0.2, B_top, width=0.2, color='k', label='overall')
plt.xticks(x_pos, ['excellent', 'good', 'fair', 'poor'])
plt.ylabel('membership')
plot_nicely()
plt.show()